In [10]:
import pandas as pd
import sqlite3

In [11]:
conn = sqlite3.connect("../data/checking-logs.sqlite")
schema = pd.read_sql("PRAGMA table_info(checker)", conn)
print(schema)

   cid       name       type  notnull dflt_value  pk
0    0      index    INTEGER        0       None   0
1    1     status       TEXT        0       None   0
2    2    success    INTEGER        0       None   0
3    3  timestamp  TIMESTAMP        0       None   0
4    4  numTrials    INTEGER        0       None   0
5    5    labname       TEXT        0       None   0
6    6        uid       TEXT        0       None   0


In [12]:
test_10 = pd.read_sql("SELECT * FROM test LIMIT 10;", conn)
print(test_10)

       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


In [13]:
df_min = pd.read_sql("""
                    SELECT uid, MIN(delta) AS delta
                    FROM (
                        SELECT 
                            t.uid, 
                            CAST((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 
                                AS INTEGER
                            ) AS delta
                    FROM test t
                    JOIN deadlines d ON t.labname = d.labs
                    WHERE NOT t.labname = 'project1')

                    """, conn)

df_min

,uid,delta
0,user_30,-202


In [14]:
df_max = pd.read_sql("""
                    SELECT uid, MAX(delta) AS delta
                    FROM (
                        SELECT 
                            t.uid, 
                            CAST((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 
                                AS INTEGER
                            ) AS delta
                    FROM test t
                    JOIN deadlines d ON t.labname = d.labs
                    WHERE NOT t.labname = 'project1')

                    """, conn)

df_max


,uid,delta
0,user_25,-2


In [15]:
df_avg = pd.read_sql("""
                    SELECT AVG(delta) AS delta
                    FROM (
                        SELECT 
                            t.uid, 
                            CAST((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 
                                AS INTEGER
                            ) AS delta
                    FROM test t
                    JOIN deadlines d ON t.labname = d.labs
                    WHERE NOT t.labname = 'project1')

                    """, conn)

df_avg

,delta
0,-89.125


In [16]:
views_diff = pd.read_sql(
    '''
        SELECT 
            p.uid,
            AVG(t.delta) AS avg_diff,
            COUNT(p.uid) AS pageviews
        FROM (
            SELECT
                t.uid,
                CAST((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24
                    AS INTEGER            
                ) AS delta
            FROM test t
            JOIN deadlines d ON t.labname = d.labs
            WHERE NOT t.labname = 'project1'
            GROUP BY t.uid
        ) AS t
        JOIN pageviews p
        WHERE p.uid LIKE 'user_%' AND p.uid = t.uid
        GROUP BY p.uid
        ORDER BY p.uid
''', conn)

views_diff


,uid,avg_diff,pageviews
0,user_1,-6.0,28
1,user_10,-39.0,89
2,user_14,-200.0,143
3,user_17,-81.0,47
4,user_18,-4.0,3
5,user_19,-148.0,16
6,user_21,-126.0,10
7,user_25,-148.0,179
8,user_28,-98.0,149
9,user_3,-75.0,317


In [17]:
correlation = views_diff[['avg_diff', 'pageviews']].corr()
print(correlation.iloc[0, 1])

-0.06296686755615415


In [18]:
conn.close()